In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import numpy.ma as ma
import pandas as pd
import scipy as sp
import seaborn as sns
from scipy import stats
from scipy.stats import bootstrap
# bayes_toolbox.glm as bg
# import arviz as az
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Set some defaults
sns.set_theme(context="paper", font_scale=1.2)
sns.set_style("ticks")
plt.rcParams["mathtext.default"] = "regular"
plt.rc("axes.spines", top=False, right=False)

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

In [ ]:
df = pd.read_csv("../results/ige_ege_jump.csv")
df["adaptation"] = np.nan
df["total error"] = np.nan
df.columns

df.head()

## Functions

In [ ]:
def remove_outliers(df, var, z_thresh, SN, TN):

    # z-score
    df[var + "_z"] = df.groupby("SN")[var].transform(stats.zscore)

    # Create outlier column
    df[var + "_outlier"] = (
        (np.abs(df[var + "_z"]) > z_thresh) 
    )

    # Calculate within-subject mean using non-outlier trials only
    df[var + "_mean"] = (
        df[(np.abs(df[var + "_z"]) <= z_thresh)]
        .groupby(SN)[var].transform("mean")
    )

    # Replace outliers with NaNs
    df[var + "_clean"] = np.where(
        (np.abs(df[var + "_z"]) > z_thresh),
        np.nan,
        df[var])

    # Create col that replaces outliers with within-subject mean values (may not use)
    df[var + "_mean"] = df.groupby(SN)[var + "_mean"].transform(lambda x: x.fillna(np.nanmean(x)))

    # Count number of outliers per participant
    num_outliers = df.groupby(SN)[var + "_outlier"].sum()
    print(num_outliers)

    # Print proportion of trials removed
    print(num_outliers / df[TN].max())

def motor_sd(df, numBaseline, cleanData, SN, TN):
    mask = (df[TN] > (numBaseline - 50)) & (df[TN] <= numBaseline)
    df["motor_sd"] = df[mask].groupby(SN)[cleanData].transform("std")
    df[mask]
    df.head()

def binning(df, var, bin_var, clean, SN):
    df[bin_var + "_quintile"] = df.groupby(["SN", var], observed=False)[clean].transform(
        lambda x: pd.qcut(x, 5, labels=range(1,6)))
    
    return df.groupby([SN, var, bin_var + "_quintile"], observed=False)[[clean, "adaptation"]].mean().reset_index()

def regress(df, var, SN):

    beta_binned_var = np.zeros(16)

    for idx, val in enumerate(df[SN].unique()):
        subj_var = df.loc[df[SN] == val, :]
        model_var = ols("adaptation ~ " + var + "- 1", subj_var).fit()
        beta_binned_var[idx] = model_var._results.params[0]

    beta_binned_var = (beta_binned_var,)
    res_var = bootstrap(beta_binned_var, np.mean, method="percentile").confidence_interval

    print(f"95\% CI for " + var + f": {res_var}")

def popn(df, var, quintile, SN):
    return df.groupby([var, quintile], as_index=False, observed=False).mean().drop(columns=SN)

In [ ]:
def plot_coeffs(ax, data, y1, y2):
    sns.pointplot(data=data, x="EGE", y=y1, ax=ax, c="r")
    sns.stripplot(data=data, x="EGE", y=y1, alpha=0.3, ax=ax, c="r")
    sns.pointplot(data=data, x="IGE", y=y2, ax=ax, c="b")
    sns.stripplot(data=data, x="IGE", y=y2, alpha=0.3, ax=ax, c="b")
    ax.axhline(linewidth=0.5, color="k")
    plt.tight_layout()
    return ax

def set_labels(fig, ax, title=None):
    ax.set(xlabel="", ylabel=r"$\beta$ (sensitivity)", title=title, ylim=(-0.25, 0.85))
    sns.despine()
    
def set_style():
    # This sets reasonable defaults for font size for
    # a figure that will go in a paper
    sns.set_theme(context="paper", font_scale=1.2)
    sns.set_style("ticks")

def plot_binned_data(binned_data, popn_data, ege_col, beta_ege_col, beta_ige_col):
    '''
    Group-level figure showing relationships between adaptation and IGE/EGE.
    '''

    # Set-up color cycles
    colors_ige = plt.get_cmap("Blues")(np.linspace(0.2, 0.8, 5))
    colors_ege = plt.get_cmap("Reds")(np.linspace(0.2, 0.8, 5))
    
    # Plot adaptive response vs IGE and EGE for rotation trials
    perts = np.unique(binned_data[ege_col])
    pert = []
    x_mean_ige = np.zeros(5)
    x_err_ige = np.zeros(5)
    y_mean_ige = np.zeros(5)
    y_err_ige = np.zeros(5)
    x_mean_ege = np.zeros(5)
    x_err_ege = np.zeros(5)
    y_mean_ege = np.zeros(5)
    y_err_ege = np.zeros(5)
    h = np.zeros(len(perts), dtype=int)
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(7, 2.75), width_ratios=[3, 3, 1])
    for i in range(len(perts)):
        idx = binned_data[ege_col] == perts[i]
        pert = pd.DataFrame(binned_data.loc[idx, :])
        pert["ige_mean"] = np.nan
        for j in range(5):
            idx_bin = pert["ige_quintile"] == j + 1
            temp = pert.loc[idx_bin, :]
            x_mean_ige[j] = pert.loc[idx_bin, "theta_maxradv_clean"].mean()
            x_err_ige[j] = pert.loc[idx_bin, "theta_maxradv_clean"].sem()
            y_mean_ige[j] = pert.loc[idx_bin, "adaptation"].mean()
            y_err_ige[j] = pert.loc[idx_bin, "adaptation"].sem()
            pert.loc[idx_bin, "ige_mean"] = pert.loc[idx_bin, "theta_maxradv_clean"].mean()
            ax2.errorbar(x=x_mean_ige[j], y=y_mean_ige[j], yerr=y_err_ige[j], ecolor=colors_ege[i],
                     **{"marker":"o", "markeredgecolor":colors_ege[i], "markerfacecolor":colors_ige[j], "linestyle":"none"}) 
            ax2.set_xticks((-4, -2, 0, 2, 4))
        sns.regplot(data=pert, x="ige_mean", y="adaptation", ax=ax2, ci=0, 
                    scatter=False, scatter_kws={"color":colors_ege[i]}, 
                    line_kws={"color":colors_ege[i], "linewidth":1})
    # Plot overall correlation and compute statistics, bootstrapped CIs
    sns.regplot(data=popn_data, x="theta_maxradv_clean", y="adaptation", ax=ax2,
                scatter=False, ci=None, line_kws={"color":"k", "linestyle":"--"})
    slope_ige, _, r_ige, p_ige, std_err_ige = stats.linregress(
        popn_data["theta_maxradv_clean"], popn_data["adaptation"])
    ax2.text(-4, 3.5, f"$r^2$={r_ige**2:.3f}", fontsize="small")
    ax2.text(-4, 3.0, f"$slope=${slope_ige:.3f}", fontsize="small")
    
    # Outer loop is for ige bins, inner loop for pert levels
    for i in range(5):
        idx = binned_data["ige_quintile"] == i + 1
        binned = pd.DataFrame(binned_data.loc[idx, :])
        for j in range(5):
            idx_ege = binned[ege_col] == perts[j]
            x_mean_ege[j] = binned.loc[idx_ege, ege_col].mean()
            x_err_ege[j] = binned.loc[idx_ege, ege_col].sem()
            y_mean_ege[j] = binned.loc[idx_ege, "adaptation"].mean()
            y_err_ege[j] = binned.loc[idx_ege, "adaptation"].sem()
            binned.loc[idx_ege, "ege_mean"] = binned.loc[idx_ege, ege_col].mean()
            ax1.errorbar(x=x_mean_ege[j], y=y_mean_ege[j], yerr=y_err_ege[j], ecolor=colors_ege[i],
                    **{"marker":"o", "markeredgecolor":colors_ege[j], "markerfacecolor":colors_ige[i], "linestyle":"none"})
            ax1.set_xticks((-4, -2, 0, 2, 4))
        sns.regplot(data=binned, x="ege_mean", y="adaptation", ax=ax1, ci=0,
                    scatter=False, line_kws={"color":colors_ige[i], "linewidth":1})
    sns.regplot(data=popn_data, x=ege_col, y="adaptation", ax=ax1,
                scatter=False, ci=None, line_kws={"color":"k", "linestyle":"--"})
    # Plot overall correlation and compute statistics, bootstrapped CIs
    r_ege, p_ege = sp.stats.pearsonr(popn_data[ege_col], popn_rotation["adaptation"])
    slope_ege, _, r_ege, p_ege, std_err_ege = stats.linregress(
        popn_data[ege_col], popn_data["adaptation"])
    ax1.text(-4, -1.5, f"$r^2$={r_ege**2:.3f}", fontsize="small")
    ax1.text(-4, -2.0, f"$slope=${slope_ege:.3f}", fontsize="small")
    
    # More figure aesthetics    
    ax1.set(xlabel="Externally-generated error ($\degree$)", ylabel="Adaptation ($\degree$)", xlim=[-4.5, 4.5], ylim=[-4, 4])
    sns.despine()
    ax2.set(xlabel="Internally-generated error ($\degree$)", ylabel="Adaptation ($\degree$)", xlim=[-4.5, 4.5], ylim=[-4, 4])
    plt.tight_layout()
    
    ax3 = plot_coeffs(ax3, df_betas, df_betas[beta_ege_col], df_betas[beta_ige_col])
    set_labels(fig, ax3)
    ax3.tick_params(axis="x", labelrotation=45)
    plt.tight_layout()

    return fig

## Now we get real

In [ ]:
# Remove Outliers
remove_outliers(df, "theta_maxradv", 3.5, "SN", "TN")
remove_outliers(df, "hand_max_dist", 3.5, "SN", "TN")

In [ ]:
motor_sd(df, 70, "theta_maxradv_clean", "SN", "TN")
df.head()

In [ ]:
# adaptation is quantified as function of perturbation -> look at how hand angle changes in response

# Choose whether to visualize all data or only adaptation trials preceded by no fb
truncate = False

adapt_rotation = pd.DataFrame()
adapt_jump = pd.DataFrame()
vmr_all = pd.DataFrame() # visual motor rotation?
sigma_motor_bsl = [] # baseline motor variability 
sigma_motor_train = [] # training motor variability 
sigma_v_train = [] # visual variability
beta_rotation_ege = np.zeros(len(df["id"].unique()))
beta_rotation_ige = np.zeros(len(df["id"].unique()))
beta_jump_ege = np.zeros(len(df["id"].unique()))
beta_jump_ige = np.zeros(len(df["id"].unique()))
beta_ext = np.zeros(len(df["id"].unique()))
# Create a new variable for movement extent errors (Change this)
df["extent_err"] = df["hand_max_dist_clean"] - 90 # don't really know why 

for k, s in enumerate(df["id"].unique()):
    # Create subject-specific data frame
    subj = df[df["id"] == s].reset_index(drop=True)
    
    # Calculate baseline variability
    sigma_motor_bsl.append(subj.loc[20:70, "theta_maxradv_clean"].std())
    
    # Create truncated data frame without baseline trials
    training_start = 70
    subj_train = subj.iloc[training_start - 1:, :].reset_index(drop=True)
    pert_idx = np.arange(1, len(subj_train), 2)
    
    # Loop through trials to get adaptation index
    for i in np.arange(1, len(subj_train), 2):
        subj_train.loc[i, "adaptation"] = subj_train.loc[i + 1, "theta_maxradv_clean"] - subj_train.loc[i - 1, "theta_maxradv_clean"] 
        subj_train.loc[i, "total error"] = subj_train.loc[i, "theta_maxradv_clean"] + subj_train.loc[i, "rotation"]
        # Measure IGE adaptation to movement extent errors
        subj_train.loc[i, "adaptation_ext"] = subj_train.loc[i + 1, "extent_err"] - subj_train.loc[i - 1, "extent_err"]
    
    # Calculate motor and vis fb variability during training
    sigma_motor_train.append(subj_train["theta_maxradv_clean"].std())
    vis_fb_idx = subj_train["fbi"] == 1
    sigma_v_train.append(subj_train.loc[vis_fb_idx, "total error"].std())

    # Dataframes for perturbation trials only (trim null trials)
    subj_adapt = subj_train.loc[pert_idx, :].reset_index()
    subj_adapt["tgt_error"] = subj_adapt["tgt_jump"] * -1
    
    # Pick out trials preceded by no-feedback null trial w/boolean mask
    nofb_idx = np.zeros(len(subj_train), dtype=bool)
    nofb_idx[1:] = subj_train.loc[0:len(subj_train) - 2, "fbi"] == 0  # .loc slicing includes start and stop index
    
    # Insert zero at start of array to index pert trials with preceding no-fb trial
    nofb_triplet_idx = nofb_idx
    subj_adapt_nofb = subj_train.loc[nofb_triplet_idx, :].reset_index()
    subj_adapt_nofb["tgt_error"] = subj_adapt_nofb["tgt_jump"] * -1
    
    # Create separate vmr and target jump data frames
    vmr = subj_adapt.loc[subj_adapt["tgt_jump"] == 0, :]  # Includes 0d rotation
    vmr_nofb = subj_adapt_nofb.loc[subj_adapt_nofb["tgt_jump"] == 0, :]
    mask_jump = (subj_adapt["tgt_jump"] != 0) | ((subj_adapt["tgt_jump"] == 0) & (subj_adapt["rotation"] == 0))
    mask_jump_nofb = (subj_adapt_nofb["tgt_jump"] != 0) | ((subj_adapt_nofb["tgt_jump"] == 0) & (subj_adapt_nofb["rotation"] == 0))
    jump = subj_adapt[mask_jump]
    jump_nofb = subj_adapt_nofb[mask_jump_nofb]
    
    # Concatenate individual subject data frames
    adapt_rotation = pd.concat([adapt_rotation, vmr], ignore_index=True)
    adapt_jump = pd.concat([adapt_jump, jump], ignore_index=True)
    
    # Assign correct data frame for visualization
    if truncate == True:
        df_vmr = vmr_nofb.copy()
        df_jump = jump_nofb.copy()
    else:
        df_vmr = vmr.copy()
        df_jump = jump.copy()
    
    # Plot data for adapt vs vmr-ege
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))
    sns.regplot(data=df_vmr, x="rotation", y="adaptation", x_jitter=0.1, 
               scatter_kws=dict(alpha=0.3, color='gray', s=20), 
               line_kws=dict(linewidth=2, color='red'), ci=95, ax=ax1)
    ax1.axhline(0, c="k", linewidth=0.5)
    ax2.axhline(0, c="k", linewidth=0.5)
    ax1.set(xticks=np.unique(df_vmr["rotation"]), xlabel="Rotation [EGE] ($\degree$)", 
            xlim=(-6.5, 6.5), yticks=np.arange(-10, 10.01, 2), ylabel="Adaptation ($\degree$)")
    sns.despine()
    
    # Plot data for adapt vs ige
    sns.regplot(data=df_vmr, x="theta_maxradv_clean", y="adaptation", x_jitter=0.1, 
               scatter_kws=dict(alpha=0.3, color='gray', s=20), ci=95, ax=ax2)
    ax2.set(xticks=np.unique(df_vmr["rotation"]), xlabel="Motor noise [IGE] ($\degree$)", 
        xlim=(-6.5, 6.5), yticks=np.arange(-10, 10.01, 2), ylabel="Adaptation ($\degree$)")
    sns.despine()
    plt.tight_layout()
    
    # Save individual data fig and print out linear regression results
    if k == 12:
        # Get rid of all rows with NaNs in either adaptation or hand cols
        df_vmr = df_vmr.dropna(subset=["adaptation", "theta_maxradv_clean"])
    
        fig.savefig("images/s13.pdf", dpi=300)
        slope_ege, _, _, _, _ = stats.linregress(x=df_vmr["rotation"], y=df_vmr["adaptation"])
        slope_ige, _, _, _, _ = stats.linregress(x=df_vmr["theta_maxradv_clean"], y=df_vmr["adaptation"])
        print(f"The slope for EGE is {slope_ege:.3f}; IGE is {slope_ige:.3f}.")

    # Plot data for adapt vs tgt_jump-ege
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))
    sns.regplot(data=df_jump, x="tgt_error", y="adaptation", x_jitter=0.1, 
               scatter_kws=dict(alpha=0.3, color='gray', s=20), 
               line_kws=dict(linewidth=2, color='red'), ci=95, ax=ax1)
    ax1.axhline(0, c="k", linewidth=0.5)
    ax2.axhline(0, c="k", linewidth=0.5)
    ax1.set(xticks=np.unique(df_jump["tgt_error"]), xlabel="Target jump [EGE] ($\degree$)", 
            xlim=(-6.5, 6.5), yticks=np.arange(-10, 10.01, 2), ylabel="Adaptation ($\degree$)")
    sns.despine()
    
    # Plot data for adapt vs tgt_jump-ige
    sns.regplot(data=df_jump, x="theta_maxradv_clean", y="adaptation", x_jitter=0.1, 
               scatter_kws=dict(alpha=0.3, color='gray', s=20), ci=95, ax=ax2)
    ax2.set(xticks=np.unique(df_jump["tgt_error"]), xlabel="Motor noise [IGE] ($\degree$)", 
        xlim=(-6.5, 6.5), yticks=np.arange(-10, 10.01, 2), ylabel="Adaptation ($\degree$)")
    sns.despine()
    plt.tight_layout()
    
    # Some simple frequentist statistics
    data1 = pd.DataFrame({
        'ege': vmr["rotation"], 
        'ige': vmr["theta_maxradv_clean"],
        'adapt': vmr["adaptation"]
    })
    
    data2 = pd.DataFrame({
        "ege": jump["tgt_error"],
        "ige": jump["theta_maxradv_clean"],
        "adapt": jump["adaptation"]
    })

    # Fit the regression models
    model_vmr = ols("adapt ~ ege + ige - 1", data1).fit()
    model_jump = ols("adapt ~ ege + ige - 1", data2).fit()
    beta_rotation_ege[k] = model_vmr._results.params[0]
    beta_rotation_ige[k] = model_vmr._results.params[1]
    beta_jump_ege[k] = model_jump._results.params[0]
    beta_jump_ige[k] = model_jump._results.params[1]
    # beta_ext[k] = model_ext._results.params[0]
    
    # Export each subject's VMR data and putting into data frame
    idx_rotation = (
        (subj_train["rotation"] != 0) 
        & (subj_train["tgt_jump"] == 0)
    ) # Non-zero rotation trials

    idx_zero = (
        (subj_train["rotation"] == 0) 
        & (subj_train["tgt_jump"] == 0) 
        & (subj_train.index.values % 2 == 1)  # null trials  
    ) # Zero deg "rotation" trials

    idx_combined = idx_rotation | idx_zero
    subj_train["perturbation"] = idx_combined
    subj_train["motor_sd"] = sigma_motor_bsl[-1]
    indices = np.flatnonzero(idx_combined)
    indices = np.sort(np.unique(np.concatenate((indices - 1, indices, indices + 1))))
    vmr_all = pd.concat([vmr_all, subj_train.loc[indices, :]], ignore_index=True)

# Save processed data
vmr_all.to_csv("../results/vmr_all.csv", index=False)

In [ ]:
rotation_binned = binning(adapt_rotation, "rotation", "ige", "theta_maxradv_clean", "SN")
jump_binned = binning(adapt_jump, "tgt_error", "ige","theta_maxradv_clean", "SN")


In [ ]:
regress(rotation_binned, "rotation", "SN")
regress(rotation_binned, "theta_maxradv_clean", "SN")
regress(jump_binned, "tgt_error", "SN")
regress(jump_binned, "theta_maxradv_clean", "SN")

In [ ]:
popn_rotation = popn(rotation_binned, "rotation", "ige_quintile", "SN")
popn_jump = popn(jump_binned, "tgt_error", "ige_quintile", "SN")  

In [ ]:
# Put regression coeffs into data frame
n = len(beta_rotation_ege)
subj_num = np.linspace(1, len(beta_rotation_ege), len(beta_rotation_ege), n)
df_betas = pd.DataFrame({
    "subject":subj_num, 
    "EGE": np.repeat("EGE", n),
    "IGE": np.repeat("IGE", n),
    "beta_rotation_ege":beta_rotation_ege * -1, 
    "beta_rotation_ige":beta_rotation_ige * -1,
    "beta_jump_ege":beta_jump_ege * -1,
    "beta_jump_ige":beta_jump_ige * -1
})

In [ ]:
# Plot binned data
fig_rotation = plot_binned_data(rotation_binned, popn_rotation, "rotation", "beta_rotation_ege", "beta_rotation_ige")
fig_jump = plot_binned_data(jump_binned, popn_jump, "tgt_error", "beta_jump_ege", "beta_jump_ige")

# Save figs
fig_rotation.savefig("images/binned-data-rotation.pdf", dpi=300)
fig_jump.savefig("images/binned-data-tgtjump.pdf", dpi=300)

# Correlation results of population-averaged (binned) data
r_ege_vmr, p_ege_vmr = sp.stats.pearsonr(popn_rotation["rotation"], popn_rotation["adaptation"])
r_ige_vmr, p_ige_vmr = sp.stats.pearsonr(popn_rotation["theta_maxradv_clean"], popn_rotation["adaptation"])
r_ege_jump, p_ege_jump = sp.stats.pearsonr(popn_jump["tgt_error"], popn_jump["adaptation"])
r_ige_jump, p_ige_jump = sp.stats.pearsonr(popn_jump["theta_maxradv_clean"], popn_jump["adaptation"])

In [ ]:
# Check difference scores of betas for normality
diff_scores_vmr = beta_rotation_ege - beta_rotation_ige
sm.qqplot(diff_scores_vmr, loc=diff_scores_vmr.mean(), scale=diff_scores_vmr.std(), line='45')
plt.show()

diff_scores_jump = beta_jump_ege - beta_jump_ige
sm.qqplot(diff_scores_jump, loc=diff_scores_jump.mean(), scale=diff_scores_jump.std(), line='45')
plt.show()

# Compute t-test on betas (flip signs when reporting to match convention set by fig)
print(stats.ttest_rel(beta_rotation_ege, beta_rotation_ige))
res_vmr = stats.ttest_1samp(diff_scores_vmr, popmean=0)
ci_vmr = res_vmr.confidence_interval(confidence_level=0.95)
print("-----------")
print("Mean Betas:")
print(f"B_rot_ege: {-(beta_rotation_ege).mean()}")
print(f"B_rot_ige: {-(beta_rotation_ige).mean()}")
print(f"B_jump_ege: {-(beta_jump_ege).mean()}")
print(f"B_jump_ige: {-(beta_jump_ige).mean()}")
print("-----------")

print("95% CIs:")
print(f"B_rot_ege:", bootstrap((-beta_rotation_ege,), np.mean, method="percentile").confidence_interval)
print(f"B_rot_ige:", bootstrap((-beta_rotation_ige,), np.mean, method="percentile").confidence_interval)
print(f"B_jump_ege:", bootstrap((-beta_jump_ege,), np.mean, method="percentile").confidence_interval)
print(f"B_jump_ige:", bootstrap((-beta_jump_ige,), np.mean, method="percentile").confidence_interval)
print("-----------")

print("One-sample tests:")
print(f"B_rot_ege: {stats.ttest_1samp(beta_rotation_ege, popmean=0)}")
print(f"B_rot_ige: {stats.ttest_1samp(beta_rotation_ige, popmean=0)}")
print(f"B_jump_ege: {stats.ttest_1samp(beta_jump_ege, popmean=0)}")
print(f"B_jump_ige: {stats.ttest_1samp(beta_jump_ige, popmean=0)}")
print("-----------")

print("Difference scores:")
print(f"Mean diff between rotation betas: {diff_scores_vmr.mean()}")
print(f"Rotation diff scores: {res_vmr}, {ci_vmr}")
print("-------------")
res_jump = stats.ttest_1samp(diff_scores_jump, popmean=0)
ci_jump = res_jump.confidence_interval(confidence_level=0.95)
print(f"Mean diff between jump betas: {diff_scores_jump.mean()}")
print(f"Jump diff scores: {res_jump}, {ci_jump}")